# Loan Analysis - Data Preprocessing

**Section Goals:**
> - Remove or fill any missing data.
> - Remove unnecessary or repetitive features.
> - Convert categorical string features to dummy variables.
> - Split data into train/test sets.
> - Remove outliers from training data.
> - Normalize/scale the features.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

In [ ]:
data = pd.read_parquet("../data/processed/01_eda_output.parquet")
print(f"Data shape: {data.shape}")
data.head()

## Missing Values

In [3]:
for column in data.columns:
    if data[column].isna().sum() != 0:
        missing = data[column].isna().sum()
        portion = (missing / data.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ==> {portion:.3f}%")

'emp_title': number of missing values '85785' ==> 6.377%
'emp_length': number of missing values '78511' ==> 5.836%
'title': number of missing values '16660' ==> 1.238%
'zip_code': number of missing values '1' ==> 0.000%
'dti': number of missing values '374' ==> 0.028%
'revol_util': number of missing values '857' ==> 0.064%


### `emp_title`

Realistically there are too many unique job titles to try to convert this to a dummy variable feature. Let's remove the `emp_title` column.

In [4]:
print(f"Unique emp_title values: {data.emp_title.nunique()}")
data.drop('emp_title', axis=1, inplace=True)

Unique emp_title values: 378353


### `emp_length`

Charge off rates are extremely similar across all employment lengths. So we are going to drop the `emp_length` column.

In [5]:
data.emp_length.unique()

array(['10+ years', '3 years', '4 years', '6 years', '7 years', '8 years',
       '2 years', '5 years', '9 years', '< 1 year', '1 year', nan],
      dtype=object)

In [6]:
for year in data.emp_length.dropna().unique():
    print(f"{year}:")
    vals = data[data.emp_length == year].loan_status.value_counts()
    print(f"  Charged Off rate: {(vals.get(0, 0) / vals.sum()) * 100:.2f}%")
    print()

10+ years:
  Charged Off rate: 18.78%

3 years:
  Charged Off rate: 19.97%

4 years:
  Charged Off rate: 19.74%

6 years:
  Charged Off rate: 19.35%

7 years:
  Charged Off rate: 19.49%

8 years:
  Charged Off rate: 19.93%

2 years:
  Charged Off rate: 19.81%

5 years:
  Charged Off rate: 19.60%

9 years:
  Charged Off rate: 19.90%

< 1 year:
  Charged Off rate: 20.53%

1 year:
  Charged Off rate: 20.56%



In [7]:
data.drop('emp_length', axis=1, inplace=True)

### `title`

The title column is simply a string subcategory/description of the purpose column. So we are going to drop the title column.

In [8]:
data.drop('title', axis=1, inplace=True)

### `mort_acc`

There are many ways we could deal with this missing data. We will group the dataframe by the `total_acc` and calculate the mean of `mort_acc` per `total_acc` entry to fill in the missing values.

In [9]:
print(f"Missing mort_acc: {data.mort_acc.isna().sum()}")
data.corr(numeric_only=True)['mort_acc'].drop('mort_acc').sort_values()

Missing mort_acc: 0


int_rate               -0.09
dti                    -0.03
revol_util              0.02
pub_rec                 0.03
pub_rec_bankruptcies    0.04
loan_status             0.07
open_acc                0.11
installment             0.16
revol_bal               0.17
annual_inc              0.17
loan_amnt               0.19
total_acc               0.28
Name: mort_acc, dtype: float64

In [10]:
total_acc_avg = data.groupby('total_acc')['mort_acc'].mean()

def fill_mort_acc(total_acc, mort_acc):
    if np.isnan(mort_acc):
        return total_acc_avg.get(total_acc, total_acc_avg.mean()).round()
    else:
        return mort_acc

data['mort_acc'] = data.apply(lambda x: fill_mort_acc(x['total_acc'], x['mort_acc']), axis=1)

### `revol_util` & `pub_rec_bankruptcies`

These two features have missing data points, but they account for less than 0.5% of the total data. So we are going to drop the rows with missing values.

In [11]:
for column in data.columns:
    if data[column].isna().sum() != 0:
        missing = data[column].isna().sum()
        portion = (missing / data.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ==> {portion:.3f}%")

'zip_code': number of missing values '1' ==> 0.000%
'dti': number of missing values '374' ==> 0.028%
'revol_util': number of missing values '857' ==> 0.064%


In [12]:
data.dropna(inplace=True)
print(f"Shape after dropping NAs: {data.shape}")

Shape after dropping NAs: (1344079, 25)


## Categorical Variables and Dummy Variables

In [13]:
print([column for column in data.columns if data[column].dtype == object])

['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'zip_code', 'addr_state', 'initial_list_status', 'application_type']


### `term`

In [14]:
data.term.unique()

array([' 36 months', ' 60 months'], dtype=object)

In [15]:
term_values = {' 36 months': 36, ' 60 months': 60}
data['term'] = data.term.map(term_values)
data.term.unique()

array([36, 60])

### `grade` & `sub_grade`

We know that `grade` is just a parent feature of `sub_grade`, so we are going to drop it.

In [16]:
data.drop('grade', axis=1, inplace=True)

In [17]:
dummies = ['sub_grade', 'verification_status', 'purpose', 'initial_list_status',
           'application_type', 'home_ownership']
data = pd.get_dummies(data, columns=dummies, drop_first=True)

### `zip_code`

The raw data already has `zip_code` as a separate column. We'll convert it to dummy variables.

In [18]:
data['zip_code'].value_counts().head(10)

zip_code
945xx    14990
750xx    14467
112xx    13818
606xx    12427
300xx    12115
331xx    11319
070xx    10748
891xx    10613
100xx    10595
900xx    10556
Name: count, dtype: int64

In [19]:
data = pd.get_dummies(data, columns=['zip_code'], drop_first=True)

### `addr_state`

We'll drop `addr_state` since we already have `zip_code` dummies which capture geographic information.

In [20]:
data.drop('addr_state', axis=1, inplace=True)

### `issue_d`

This would be data leakage — we wouldn't know beforehand whether or not a loan would be issued when using our model, so in theory we wouldn't have this feature. Let's drop it.

In [21]:
data.drop('issue_d', axis=1, inplace=True)

### `earliest_cr_line`

This appears to be a historical timestamp feature. Extract the year from this feature.

In [22]:
data['earliest_cr_line'] = data.earliest_cr_line.dt.year
print(f"Unique years: {data.earliest_cr_line.nunique()}")
data.earliest_cr_line.value_counts().head(10)

Unique years: 72


earliest_cr_line
2001    88824
2002    85909
2000    85360
2003    85340
2004    82777
1999    77155
2005    74088
1998    65185
2006    63462
1997    55867
Name: count, dtype: int64

## Train Test Split

In [23]:
w_p = data.loan_status.value_counts()[0] / data.shape[0]
w_n = data.loan_status.value_counts()[1] / data.shape[0]

print(f"Weight of positive values (Charged Off): {w_p:.4f}")
print(f"Weight of negative values (Fully Paid): {w_n:.4f}")

Weight of positive values (Charged Off): 0.1996
Weight of negative values (Fully Paid): 0.8004


In [24]:
train, test = train_test_split(data, test_size=0.33, random_state=42)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (900532, 1011)
Test shape: (443547, 1011)


## Removing Outliers

We remove outliers only from the training set to avoid data leakage.

In [25]:
print(f"Train shape before outlier removal: {train.shape}")
train = train[train['annual_inc'] <= 250000]
train = train[train['dti'] <= 50]
train = train[train['open_acc'] <= 40]
train = train[train['total_acc'] <= 80]
train = train[train['revol_util'] <= 120]
train = train[train['revol_bal'] <= 250000]
print(f"Train shape after outlier removal: {train.shape}")

Train shape before outlier removal: (900532, 1011)
Train shape after outlier removal: (887550, 1011)


## Normalizing the Data

In [26]:
X_train, y_train = train.drop('loan_status', axis=1), train.loan_status
X_test, y_test = test.drop('loan_status', axis=1), test.loan_status

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (887550, 1010), y_train: (887550,)
X_test: (443547, 1010), y_test: (443547,)


In [27]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
# Convert to float32 for model compatibility
X_train_scaled = np.array(X_train_scaled).astype(np.float32)
X_test_scaled = np.array(X_test_scaled).astype(np.float32)
y_train = np.array(y_train).astype(np.float32)
y_test = np.array(y_test).astype(np.float32)

print(f"X_train dtype: {X_train_scaled.dtype}")
print(f"y_train dtype: {y_train.dtype}")

X_train dtype: float32
y_train dtype: float32


In [29]:
# Save processed data
np.savez("../data/processed/02_processed_data.npz",
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train, y_test=y_test)
joblib.dump(scaler, "../data/processed/02_scaler.joblib")

print("Saved processed data and scaler.")
print(f"Number of features: {X_train_scaled.shape[1]}")

Saved processed data and scaler.
Number of features: 1010
